### Importando bibliotecas


In [160]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### Criando tabelas

In [161]:

patients = pd.read_csv('hosp/patients.csv') # 100 pacientes
icu_stays = pd.read_csv('icu/icustays.csv') # Todos os 100 ficaram na UTI
input_events = pd.read_csv('icu/inputevents.csv') # 2 Não receberam medicamentos
procedure_events = pd.read_csv('icu/procedureevents.csv') # 2 Não receberam procedimentos
output_events = pd.read_csv('icu/outputevents.csv')
ingredient_events = pd.read_csv('icu/ingredientevents.csv')
items = pd.read_csv('icu/d_items.csv') # Itens e seus códigos

print("Nº de pacientes:", patients.shape[0])

Nº de pacientes: 100


### Contagem de eventos

#### Vasopressores (unidades de medida bem diferentes, talvez compense mais usar o número total de eventos?)

In [173]:
vasopressores = ['Vasopressin',
                 'Dopamine',
                 'Epinephrine',
                 'Norepinephrine',
                 'Phenylephrine',
                 'Milrinone',
                 'Dobutamine']


vp = items[items['label'].isin(vasopressores)]

vp = pd.merge(input_events,vp)
vasopres = vp.groupby(['subject_id','stay_id','itemid','label','amountuom'])['itemid'].count().reset_index(name='event_count')
vasopres['stay_total_amount'] = vp.groupby(['subject_id','stay_id','itemid','label','amountuom'])['amount'].sum().values
vasopres

,subject_id,stay_id,itemid,label,amountuom,event_count,stay_total_amount
0,10002428,35479615,221906,Norepinephrine,mg,31,13.224821
1,10002428,38875437,221662,Dopamine,mg,11,196.758289
2,10002428,38875437,221749,Phenylephrine,mg,7,54.767366
3,10002428,38875437,221906,Norepinephrine,mg,9,0.671676
4,10002495,36753294,221662,Dopamine,mg,3,53.395216
...,...,...,...,...,...,...,...
85,10039708,33281088,221906,Norepinephrine,mg,174,264.498800
86,10039708,33281088,222315,Vasopressin,units,9,327.040000
87,10039708,38559363,221906,Norepinephrine,mg,5,2.249693
88,10040025,36107367,221749,Phenylephrine,mg,13,52.036688


#### Contagem de eventos de ventilação (talvez dê pra usar a duração em minutos??)

In [179]:
ventilacao = ['Mask Ventilation (Intubation)',
              'Non-invasive Ventilation',
              'Invasive Ventilation']

vt = items[items['label'].isin(ventilacao)]
vt = pd.merge(procedure_events,vt)
vent = vt.groupby(['subject_id','stay_id','itemid','label','valueuom'])['itemid'].count().reset_index(name='event_count')
vent['total_duration'] = vt.groupby(['subject_id','stay_id','itemid','label','valueuom'])['value'].sum().values
vent

,subject_id,stay_id,itemid,label,valueuom,event_count,total_duration
0,10002428,34807493,225794,Non-invasive Ventilation,min,1,1809.0
1,10002428,35479615,225792,Invasive Ventilation,min,1,12640.0
2,10002428,38875437,225792,Invasive Ventilation,min,1,4135.0
3,10002495,36753294,225794,Non-invasive Ventilation,min,1,321.0
4,10003400,32128372,225792,Invasive Ventilation,min,1,3760.0
...,...,...,...,...,...,...,...
63,10038933,32166508,225792,Invasive Ventilation,min,1,1453.0
64,10038992,37127068,225792,Invasive Ventilation,min,1,390.0
65,10038999,39711498,225792,Invasive Ventilation,min,1,9621.0
66,10039708,33281088,225792,Invasive Ventilation,min,3,6650.0


#### Contagem de eventos de Terapia de Substituição Renal

In [164]:
tsr_ids = [ 226118,
            227357,
            225725,
            226499,
            224154,
            225810,
            225959,
            227639,
            225183,
            227438,
            224191,
            225806,
            225807,
            228004,
            228005,
            228006,
            224144,
            224145,
            224149,
            224150,
            224151,
            224152,
            224153,
            224404,
            224406,
            226457,
            224135,
            224139,
            224146,
            225323,
            225740,
            225776,
            225951,
            225952,
            225953,
            225954,
            225956,
            225958,
            225961,
            225963,
            225965,
            225976,
            225977,
            227124,
            227290,
            227638,
            227640,
            227753,
            227536,
            227525,
            225441,
            225802,
            225803,
            225805,
            225809,
            225955,
            224270,
            225436
]



vtsr = items[items['itemid'].isin(tsr_ids)]

##### Eventos de entrada (Também tem unidades de medida diferentes)

In [180]:
vtsri = pd.merge(input_events,vtsr)
tsri = vtsri.groupby(['subject_id','stay_id','itemid','label','amountuom'])['itemid'].count().reset_index(name='count')
tsri['stay_total_amount'] = vtsri.groupby(['subject_id','stay_id','itemid','label','amountuom'])['amount'].sum().values

tsri

,subject_id,stay_id,itemid,label,amountuom,count,stay_total_amount
0,10004235,34100191,227525,Calcium Gluconate (CRRT),grams,9,62.996110
1,10004235,34100191,227536,KCl (CRRT),mEq.,21,162.394674
2,10007818,32359580,227525,Calcium Gluconate (CRRT),grams,20,194.071824
3,10007818,32359580,227536,KCl (CRRT),mEq.,33,245.243769
4,10035631,30932571,227525,Calcium Gluconate (CRRT),grams,22,225.171767
5,10039708,33281088,227525,Calcium Gluconate (CRRT),grams,18,166.606406
6,10039708,33281088,227536,KCl (CRRT),mEq.,51,421.719093


##### Eventos de procedimento (unidades diferentes, mas todas são de tempo)

In [182]:

vtsrp = pd.merge(procedure_events,vtsr)
tsrp = vtsrp.groupby(['subject_id','stay_id','itemid','label','valueuom'])['itemid'].count().reset_index(name='count')
tsrp['stay_total_duration'] = vtsrp.groupby(['subject_id','stay_id','itemid','label','valueuom'])['value'].sum().values
tsrp

,subject_id,stay_id,itemid,label,valueuom,count,stay_total_duration
0,10004235,34100191,224270,Dialysis Catheter,min,1,5261.000000
1,10004235,34100191,225802,Dialysis - CRRT,min,1,2367.000000
2,10007818,32359580,224270,Dialysis Catheter,min,3,26363.000000
3,10007818,32359580,225441,Hemodialysis,hour,1,3.500000
4,10007818,32359580,225441,Hemodialysis,min,2,453.000000
5,10007818,32359580,225802,Dialysis - CRRT,day,1,1.645833
6,10007818,32359580,225802,Dialysis - CRRT,min,3,5217.000000
7,10010471,32119961,224270,Dialysis Catheter,min,1,707.000000
8,10010471,32119961,225441,Hemodialysis,min,1,131.000000
9,10021938,33083787,225441,Hemodialysis,min,1,1883.000000
